In [13]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, max_error
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import shap
from config import DATA_DIR
from ipywidgets import Dropdown, Button, Output, VBox, HBox, Label, Layout, IntText
from IPython.display import display, clear_output
import json
import optuna

In [2]:
from src.modelling.build_features_for_cv_folds import build_features_for_cv_folds
from src.modelling.build_features_for_range import build_features_for_range

In [3]:
df_parkings = pd.read_parquet(DATA_DIR/'parkings.parquet')
df_parkings_availabilities = pd.read_parquet(DATA_DIR/'parking_availabilities.parquet')
df_sks_users = pd.read_parquet(DATA_DIR/'sks_users.parquet')
df_calendar = pd.read_parquet(DATA_DIR/'calendar.parquet')

In [4]:
# df_parkings = pd.read_parquet(DATA_DIR/'parkings2.parquet').drop(columns=['access', 'external_id']) #new data has additional columnswitch break the pipline
# df_parkings_availabilities = pd.read_parquet(DATA_DIR/'parking_availabilities2.parquet')
# df_sks_users = pd.read_parquet(DATA_DIR/'sks_users2.parquet')
# df_parkings_availabilities['measured_at'] = pd.to_datetime(
#     df_parkings_availabilities['measured_at'],
#     format='ISO8601'
# )
# df_sks_users['external_timestamp'] = pd.to_datetime(
#     df_sks_users['external_timestamp'],
#     format='ISO8601'
# )


In [5]:
df_min = df_parkings_availabilities.groupby('parking_id')['spaces_left'].max()
df_min.name = 'max_spaces_left'
df_parkings = df_parkings.merge(df_min, left_on='id', right_index=True)

print(df_parkings[['name', 'places', 'max_spaces_left']])

                 name  places  max_spaces_left
0        Architektura      75               84
1  Parking Wrońskiego     207              207
2             Polinka      54               61
3           D20 - D21      76               49
4  GEO LO1 Geocentrum     301              267


In [6]:
# Manualy collected distances based on google maps
manual_data = [
    {'id': 7, 'd_to_7': 0,    'd_to_4': 2600, 'd_to_2': 2100, 'd_to_5': 2100, 'd_to_6': 2500, 'dist_to_sks': 2500},
    {'id': 4, 'd_to_7': 2600, 'd_to_4': 0,    'd_to_2': 800,  'd_to_5': 350,  'd_to_6': 1800, 'dist_to_sks': 42},
    {'id': 2, 'd_to_7': 2100, 'd_to_4': 500,  'd_to_2': 0,    'd_to_5': 650,  'd_to_6': 1800, 'dist_to_sks': 450},
    {'id': 5, 'd_to_7': 2100, 'd_to_4': 350,  'd_to_2': 650,  'd_to_5': 0,    'd_to_6': 2400, 'dist_to_sks': 400},
    {'id': 6, 'd_to_7': 2500, 'd_to_4': 1800, 'd_to_2': 1800, 'd_to_5': 2400, 'd_to_6': 0,    'dist_to_sks': 1500}
]

processed_rows = []

for entry in manual_data:
    current_id = entry['id']
    
    # Create a simple list of neighbors: (distance, neighbor_id)
    neighbors = []
    for key, dist in entry.items():
        if key.startswith('d_to_') and key != f'd_to_{current_id}':
            neighbor_id = int(key.split('_')[-1]) # Extract 4 from 'd_to_4'
            neighbors.append((dist, neighbor_id))
            
    # Sort closest to farthest
    neighbors.sort()
    
    # Build the simple row
    processed_rows.append({
        'id': current_id,
        'closest_id': neighbors[0][1],
        'second_closest_id': neighbors[1][1],
        'third_closest_id': neighbors[2][1],
        'fourth_closest_id': neighbors[3][1],
        'distance_to_sks': entry['dist_to_sks']
    })

df_features = pd.DataFrame(processed_rows)
df_parkings = df_parkings.merge(df_features, on='id', how='left')

# Check result
df_parkings.head()

,id,symbol,type,name,open_hour,close_hour,places,geo_lan,geo_lat,is_active,is_visible,address,created_at,updated_at,max_spaces_left,closest_id,second_closest_id,third_closest_id,fourth_closest_id,distance_to_sks
0,7,E01,O,Architektura,06:00:00,22:30:00,75,17.054167,51.118736,True,True,"Bolesława Prusa 53/55, 50-317 Wrocław\r\n",2025-02-03 07:51:16.241000+00:00,2025-12-06 19:46:06.731000+00:00,84,2,5,6,4,2500
1,4,WRO,O,Parking Wrońskiego,06:00:00,22:00:00,207,17.055565,51.108963,True,True,"Hoene-Wrońskiego 10, 50-376 Wrocław",2025-02-03 07:51:16.185000+00:00,2025-12-06 19:46:06.218000+00:00,207,5,2,6,7,42
2,2,C13,O,Polinka,None,None,54,17.058468,51.107390,True,True,"wybrzeże Stanisława Wyspiańskiego 25, 50-370 W...",2025-02-03 07:51:16.210000+00:00,2025-12-06 19:46:06.343000+00:00,61,4,5,6,7,450
3,5,D20,O,D20 - D21,06:00:00,22:30:00,76,17.059677,51.110050,True,True,"Janiszewskiego 8, 50-372 Wrocław\r\n",2025-02-03 07:51:16.222000+00:00,2025-12-06 19:46:06.478000+00:00,49,4,2,7,6,400
4,6,GEO-L,O,GEO LO1 Geocentrum,06:00:00,22:30:00,301,17.055334,51.104164,True,True,"Na Grobli 15, 50-421 Wrocław\r\n",2025-02-03 07:51:16.232000+00:00,2025-12-06 19:46:06.602000+00:00,267,2,4,5,7,1500


In [7]:


DATA_START   = "2025-04-03"
DATA_END     = "2025-12-01"
LAGS         = (7, 14, 28)
ROLL_LAGS    = (7, 14, 28)
ROLL_WINDOWS = (7, 14)
FREQ         = "D"

start = pd.to_datetime(DATA_START)
end   = pd.to_datetime(DATA_END)
total_days = (end - start).days + 1
train_days = int(total_days * 0.70)
val_days   = int(total_days * 0.15)

train_start = start
train_end   = start + pd.Timedelta(days=train_days - 1)
val_start   = train_end + pd.Timedelta(days=1)
val_end     = val_start + pd.Timedelta(days=val_days - 1)
test_start  = val_end + pd.Timedelta(days=1)
test_end    = end

fmt = "%Y-%m-%d"
print(f"Train: {train_start:%Y-%m-%d} -> {train_end:%Y-%m-%d}  ({train_days} d)")
print(f"Val  : {val_start:%Y-%m-%d} -> {val_end:%Y-%m-%d}  ({val_days} d)")
print(f"Test : {test_start:%Y-%m-%d} -> {test_end:%Y-%m-%d}  ({(test_end - test_start).days + 1} d)")

# Build features: fit pipeline on train, transform val/test 
fe_kwargs = dict(
    base_df=df_parkings_availabilities,
    df_sks_users=df_sks_users,
    df_parkings=df_parkings,
    df_calendar=df_calendar,
    lags=LAGS, roll_lags=ROLL_LAGS, roll_windows=ROLL_WINDOWS, freq=FREQ,
)

train_df, fitted_pipe = build_features_for_range(
    **fe_kwargs,
    start_date=train_start.strftime(fmt), end_date=train_end.strftime(fmt),
    return_pipeline=True, fitted_pipeline=None,   # fit on train only
)
val_df = build_features_for_range(
    **fe_kwargs,
    start_date=val_start.strftime(fmt), end_date=val_end.strftime(fmt),
    fitted_pipeline=fitted_pipe,                  # transform only
)
test_df = build_features_for_range(
    **fe_kwargs,
    start_date=test_start.strftime(fmt), end_date=test_end.strftime(fmt),
    fitted_pipeline=fitted_pipe,                  # transform only
)
print(f"\nrows  train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

#Prepare matrices
target_col = 'spaces_left'
cols_to_drop = ['measured_at', target_col]
for df in (train_df, val_df, test_df):
    df['parking_id'] = df['parking_id'].astype('category')

X_train, y_train = train_df.drop(columns=cols_to_drop), train_df[target_col]
X_val,   y_val   = val_df.drop(columns=cols_to_drop),   val_df[target_col]
X_test,  y_test  = test_df.drop(columns=cols_to_drop),  test_df[target_col]


#Optuna search
def objective(trial):
    params = {
        'objective': 'reg:squarederror',
        'enable_categorical': True,
        'tree_method': 'hist',
        'n_estimators': 2000,
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'gamma':            trial.suggest_float('gamma', 1e-8, 5.0, log=True),
        'early_stopping_rounds': 50,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return mean_absolute_error(y_val, model.predict(X_val))


study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print("\nBest validation MAE:", round(study.best_value, 4))
print("Best params:", best_params)

# Persist so you don't have to retune after a kernel restart
with open("best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

# ----- Held-out test sanity check (no refit on val to keep it simple & honest) -----
final_model = xgb.XGBRegressor(
    **best_params,
    n_estimators=2000,
    objective='reg:squarederror',
    enable_categorical=True,
    tree_method='hist',
    early_stopping_rounds=50,
)
final_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
test_pred = final_model.predict(X_test)
print(f"\nTest MAE : {mean_absolute_error(y_test, test_pred):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, test_pred)):.4f}")

Train: 2025-04-03 -> 2025-09-19  (170 d)
Val  : 2025-09-20 -> 2025-10-25  (36 d)
Test : 2025-10-26 -> 2025-12-01  (37 d)


/app/src/modelling/build_features_for_range.py:91: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measured_at"] = pd.to_datetime(df["measured_at"])
/app/src/modelling/build_features_for_range.py:91: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measured_at"] = pd.to_datetime(df["measured_at"])
/app/src/modelling/build_features_for_range.py:91: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the ca


rows  train=240045  val=50255  test=47370


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-05-09 00:26:07,066] Trial 0 finished with value: 6.994856357574463 and parameters: {'learning_rate': 0.030710573677773714, 'max_depth': 10, 'min_child_weight': 8, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'reg_alpha': 2.5348407664333426e-07, 'reg_lambda': 3.3323645788192616e-08, 'gamma': 0.3426417745118369}. Best is trial 0 with value: 6.994856357574463.
[I 2026-05-09 00:26:09,468] Trial 1 finished with value: 6.864001274108887 and parameters: {'learning_rate': 0.06054365855469246, 'max_depth': 8, 'min_child_weight': 1, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'reg_alpha': 8.148018307012941e-07, 'reg_lambda': 4.329370014459266e-07, 'gamma': 3.939402261362697e-07}. Best is trial 1 with value: 6.864001274108887.
[I 2026-05-09 00:26:14,657] Trial 2 finished with value: 6.330699443817139 and parameters: {'learning_rate': 0.024878734419814436, 'max_depth': 7, 'min_child_weight': 5, 'subsample': 0.7164916560792167, 'colsa

In [8]:
optuna.visualization.plot_optimization_history(study)

Best params were found before trial 20 no point in further tuning

In [9]:
optuna.visualization.plot_param_importances(study)

In [10]:
# Load tuned params (from memory if Cell with optuna study just ran, else from disk)
try:
    best_params
except NameError:
    with open("best_params.json") as f:
        best_params = json.load(f)
print(f"Using tuned params: {best_params}")

cv_results = build_features_for_cv_folds(
    df_parkings_availabilities, df_sks_users, df_parkings, df_calendar,
    "2025-04-03", "2025-12-01", lags=(7, 14, 28), min_train_periods=90,
)

target_col = 'spaces_left'
cols_to_drop = ['measured_at', target_col]

fold_artifacts = {}
fold_results_dfs = []

for fold_data in cv_results:
    fold_idx = fold_data['fold']
    print(f"--- Training Fold {fold_idx} ---")

    train_df = fold_data['train_df'].copy()
    val_df = fold_data['val_df'].copy()
    train_df['parking_id'] = train_df['parking_id'].astype('category')
    val_df['parking_id'] = val_df['parking_id'].astype('category')

    X_train = train_df.drop(columns=cols_to_drop)
    y_train = train_df[target_col]
    X_val = val_df.drop(columns=cols_to_drop)
    y_val = val_df[target_col]

    model = xgb.XGBRegressor(
        **best_params,
        n_estimators=2000,
        objective='reg:squarederror',
        enable_categorical=True,
        tree_method='hist',
        early_stopping_rounds=50,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    predictions = model.predict(X_val)
    results_df = val_df[['measured_at', 'parking_id', target_col]].copy()
    results_df['predicted_spaces'] = predictions
    results_df['fold'] = fold_idx
    results_df['abs_error'] = (results_df[target_col] - results_df['predicted_spaces']).abs()
    results_df = results_df.reset_index(drop=True)
    fold_results_dfs.append(results_df)

    fold_artifacts[fold_idx] = {
        'model': model,
        'X_val': X_val.reset_index(drop=True),
        'y_val': y_val.reset_index(drop=True),
        'results_df': results_df,
        'explainer': shap.TreeExplainer(model),
        'feature_names': list(X_val.columns),
    }

    worst_idx = int(results_df['abs_error'].idxmax())
    print(f"  best_iter={model.best_iteration}  max|err|: "
          f"actual={results_df.iloc[worst_idx][target_col]:.2f}, "
          f"pred={results_df.iloc[worst_idx]['predicted_spaces']:.2f}")

all_predictions_df = pd.concat(fold_results_dfs, ignore_index=True)
print(f"\nStored {len(fold_artifacts)} folds in `fold_artifacts`.")

Using tuned params: {'learning_rate': 0.10550479230756851, 'max_depth': 5, 'min_child_weight': 7, 'subsample': 0.7510517106468894, 'colsample_bytree': 0.722497702919922, 'reg_alpha': 1.818170000095164e-05, 'reg_lambda': 0.2611730904137153, 'gamma': 0.07979408130869706}


/app/src/modelling/build_features_for_range.py:91: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measured_at"] = pd.to_datetime(df["measured_at"])
/app/src/modelling/build_features_for_range.py:91: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measured_at"] = pd.to_datetime(df["measured_at"])
/app/src/modelling/build_features_for_range.py:91: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the ca

--- Training Fold 0 ---
  best_iter=128  max|err|: actual=7.00, pred=86.88
--- Training Fold 1 ---
  best_iter=101  max|err|: actual=207.00, pred=100.26
--- Training Fold 2 ---
  best_iter=106  max|err|: actual=267.00, pred=113.55

Stored 3 folds in `fold_artifacts`.


In [ ]:

def make_shap_widget(fold_artifacts, target_col='spaces_left'):
    folds = sorted(fold_artifacts.keys())
    style = {'description_width': '70px'}
    dd_layout = Layout(width='180px')

    # Widgets 
    fold_dd    = Dropdown(options=folds, description='Fold:',    style=style, layout=dd_layout)
    parking_dd = Dropdown(description='Parking:', style=style, layout=dd_layout)
    year_dd    = Dropdown(description='Year:',    style=style, layout=dd_layout)
    month_dd   = Dropdown(description='Month:',   style=style, layout=dd_layout)
    day_dd     = Dropdown(description='Day:',     style=style, layout=dd_layout)
    hour_dd    = Dropdown(description='Hour:',    style=style, layout=dd_layout)
    minute_dd  = Dropdown(description='Minute:',  style=style, layout=dd_layout)

    min_label  = Label('Min selectable: -',
                       layout=Layout(width='340px'))
    max_label  = Label('Max selectable: -',
                       layout=Layout(width='340px'))
    info_label = Label('')
    refresh_btn = Button(description='Refresh SHAP', button_style='primary',
                         icon='refresh', layout=Layout(width='160px'))
    out = Output()

    # Suppresses observe handlers while we programmatically cascade
    _busy = {'flag': False}

    def filtered(up_to: int) -> pd.DataFrame:
        """Return current-fold val rows filtered by selectors up to a given level.
        0=fold only, 1=+parking, 2=+year, 3=+month, 4=+day, 5=+hour, 6=+minute."""
        df = fold_artifacts[fold_dd.value]['results_df'].copy()
        df['ts'] = pd.to_datetime(df['measured_at'])
        if up_to >= 1 and parking_dd.value is not None:
            df = df[df['parking_id'] == parking_dd.value]
        if up_to >= 2 and year_dd.value is not None:
            df = df[df['ts'].dt.year == year_dd.value]
        if up_to >= 3 and month_dd.value is not None:
            df = df[df['ts'].dt.month == month_dd.value]
        if up_to >= 4 and day_dd.value is not None:
            df = df[df['ts'].dt.day == day_dd.value]
        if up_to >= 5 and hour_dd.value is not None:
            df = df[df['ts'].dt.hour == hour_dd.value]
        if up_to >= 6 and minute_dd.value is not None:
            df = df[df['ts'].dt.minute == minute_dd.value]
        return df

    def refresh_minmax():
        df = filtered(1)  # min/max for the current parking inside the fold
        if df.empty:
            min_label.value = "Min selectable: -"
            max_label.value = "Max selectable: -"
        else:
            fmt = "%Y-%m-%d %H:%M:%S %Z"
            min_label.value = f"Min selectable: {df['ts'].min().strftime(fmt).strip()}"
            max_label.value = f"Max selectable: {df['ts'].max().strftime(fmt).strip()}"

    def cascade(start_level: int):
        """Update dropdowns starting from `start_level` (1=parking ... 6=minute)."""
        _busy['flag'] = True
        try:
            if start_level <= 1:
                opts = sorted(filtered(0)['parking_id'].unique().tolist())
                parking_dd.options = opts
                if opts: parking_dd.value = opts[0]
            refresh_minmax()
            if start_level <= 2:
                opts = sorted(filtered(1)['ts'].dt.year.unique().tolist())
                year_dd.options = opts
                if opts: year_dd.value = opts[0]
            if start_level <= 3:
                opts = sorted(filtered(2)['ts'].dt.month.unique().tolist())
                month_dd.options = opts
                if opts: month_dd.value = opts[0]
            if start_level <= 4:
                opts = sorted(filtered(3)['ts'].dt.day.unique().tolist())
                day_dd.options = opts
                if opts: day_dd.value = opts[0]
            if start_level <= 5:
                opts = sorted(filtered(4)['ts'].dt.hour.unique().tolist())
                hour_dd.options = opts
                if opts: hour_dd.value = opts[0]
            if start_level <= 6:
                opts = sorted(filtered(5)['ts'].dt.minute.unique().tolist())
                minute_dd.options = opts
                if opts: minute_dd.value = opts[0]
        finally:
            _busy['flag'] = False

    #Observers (no-ops while cascading)
    def on_fold(*_):    
        if _busy['flag']: return
        cascade(1)
    def on_parking(*_): 
        if _busy['flag']: return
        cascade(2)
    def on_year(*_):    
        if _busy['flag']: return
        cascade(3)
    def on_month(*_):   
        if _busy['flag']: return
        cascade(4)
    def on_day(*_):     
        if _busy['flag']: return
        cascade(5)
    def on_hour(*_):    
        if _busy['flag']: return
        cascade(6)

    fold_dd.observe(on_fold,       names='value')
    parking_dd.observe(on_parking, names='value')
    year_dd.observe(on_year,       names='value')
    month_dd.observe(on_month,     names='value')
    day_dd.observe(on_day,         names='value')
    hour_dd.observe(on_hour,       names='value')

    def on_refresh(_):
        with out:
            clear_output(wait=True)
            art = fold_artifacts[fold_dd.value]
            df = filtered(6)
            if df.empty:
                info_label.value = "⚠ No matching row in this fold's val set."
                return
            idx = int(df.index[0])
            row = art['results_df'].iloc[idx]
            info_label.value = (
                f"actual={row[target_col]:.2f} | "
                f"pred={row['predicted_spaces']:.2f} | "
                f"|err|={row['abs_error']:.2f}"
            )
            sv = art['explainer'](art['X_val'].iloc[[idx]])
            shap.plots.waterfall(sv[0], max_display=15, show=False)
            plt.gcf().set_size_inches(10, 6)
            plt.tight_layout()
            plt.show()

    refresh_btn.on_click(on_refresh)

    cascade(1)

    return VBox([
        HBox([min_label, max_label]),
        fold_dd,
        parking_dd,
        HBox([year_dd, month_dd, day_dd]),
        HBox([hour_dd, minute_dd]),
        HBox([refresh_btn, info_label]),
        out,
    ])


display(make_shap_widget(fold_artifacts))

We got some messy predictions sometimes. Even though min free spaces should be 0, we had some negative values. We also have a some of terrible predictions when some event causes anomalies in the data. Overall model pperforms decently, better than simple algorithms like bucketing but it's max error is high and requires attention. 

In [ ]:
def get_metrics(predictions_df: pd.DataFrame, target_col: str):
    mae = mean_absolute_error(predictions_df[target_col], predictions_df['predicted_spaces'])
    mse = mean_squared_error(predictions_df[target_col], predictions_df['predicted_spaces'])
    r2 = r2_score(predictions_df[target_col], predictions_df['predicted_spaces'])
    max_err = max_error(predictions_df[target_col], predictions_df['predicted_spaces'])
    
    return {
        'MAE': mae,
        'MSE': mse,
        'R2': r2,
        'Max Error': max_err
    }
metrics = get_metrics(all_predictions_df, target_col)
for metric_name, metric_value in metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

In [ ]:
all_predictions_df['error'] = all_predictions_df[target_col] - all_predictions_df['predicted_spaces']

In [ ]:
all_predictions_df.loc[all_predictions_df['error'].abs().idxmax()]  

In [ ]:
df_parkings.head()

In [ ]:
parking_lots = all_predictions_df['parking_id'].unique()

fig, axes = plt.subplots(nrows=len(parking_lots), ncols=1, figsize=(16, 25), sharex=True)
for i, lot_id in enumerate(parking_lots):
    
    lot_data = all_predictions_df[all_predictions_df['parking_id'] == lot_id]
    
    ax = axes[i]
    
    sns.lineplot(data=lot_data, x='measured_at', y='error', ax=ax, color='red', label='Error')
    
    # Formatting to make it look professional
    max_spaces_left = df_parkings[df_parkings['id'] == lot_id]['max_spaces_left'].values[0]
    ax.set_title(f'Parking Lot ID: {lot_id}, max error: {lot_data["error"].max():.2f}, max_spaces: {max_spaces_left:.0f}', fontsize=16, fontweight='bold')
    ax.set_ylabel('Spaces')
    ax.grid(True, alpha=0.3)
    

    if i == 0:
        ax.legend(loc='upper right')
    else:
        ax.get_legend().remove()

axes[-1].set_xlabel('Time', fontsize=14)

plt.tight_layout()
plt.show()